# LAB 05: HOG-Based Industrial Defect Detection and Classification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheBilalButt/lab-05-hog-defect-detection/blob/main/Lab5_HOG_Industrial_Defect_Detection.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/TheBilalButt/lab-05-hog-defect-detection)
[![Course](https://img.shields.io/badge/Course-Computer%20Vision-blueviolet)](https://github.com/TheBilalButt)
[![Python](https://img.shields.io/badge/Python-3.10%20%7C%203.11%20%7C%203.12-blue?logo=python)](https://www.python.org/)
[![OpenCV](https://img.shields.io/badge/OpenCV-5.0.0-green?logo=opencv)](https://opencv.org/)

**Author:** Bilal Butt ([@TheBilalButt](https://github.com/TheBilalButt))  
**Institution:** Department of Computer Science / Artificial Intelligence  
**Repository:** [TheBilalButt / lab-05-hog-defect-detection](https://github.com/TheBilalButt/lab-05-hog-defect-detection)  
**Benchmark Dataset:** Northeastern University (NEU) Surface Defect Database  

---

## 1. Executive Summary & Problem Formulation

In modern high-speed industrial manufacturing (e.g., hot-rolled steel strip rolling mills, automated electronics assembly, and aerospace fabrication), manual visual inspection is strictly unviable. Human optical quality audits are constrained by operator visual fatigue, subjective bias, low throughput (typically < 2 meters/sec), and an unacceptable false escape rate exceeding 15%.

The objective of this laboratory is to design, implement, and benchmark an automated visual quality control system leveraging the **Histogram of Oriented Gradients (HOG)** feature descriptor coupled with statistical machine-learning classifiers (**Support Vector Machines** and **Random Forests**).

### Architectural Pipeline
$$\text{Surface Image} \longrightarrow \text{Grayscale & Resizing (128}\times\text{128)} \longrightarrow \text{HOG Feature Extraction} \longrightarrow \text{ML Classifier (SVM / RF)} \longrightarrow \text{QC Decision Gate}$$

In [1]:
# Environment & Dependency Setup
import os
import glob
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.feature import hog
from skimage import exposure
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold

print('Environment initialized successfully.')

## 2. Dataset Ingestion & Surface Defect Morphology

The benchmark dataset contains surface images of hot-rolled steel plates spanning six critical industrial defect categories, alongside pristine defect-free reference plates:
1. **Crazing (`Cr`)**: Network microcracking arising from thermal shock and fatigue stress.
2. **Inclusion (`In`)**: Non-metallic impurities (oxide slags) embedded into the steel matrix.
3. **Patches (`Pa`)**: Localized surface scale build-ups and plate thickness inconsistencies.
4. **Pitted Surface (`PS`)**: Cavity clusters created by chemical corrosion or localized mechanical damage.
5. **Rolled-in Scale (`RS`)**: Iron oxide scale pressed into the strip during hot continuous rolling.
6. **Scratches (`Sc`)**: Severe abrasive longitudinal grooves caused by mechanical guide friction.
7. **Normal / Defect-Free (`Normal`)**: Homogeneous rolling grain without fissures, pits, or grooves.

In [2]:
# Load industrial dataset and verify class distributions
data_dir = 'data'
image_paths = sorted(glob.glob(os.path.join(data_dir, '*.jpg')))
print(f'Total available industrial surface images: {len(image_paths)}')

images, binary_labels, multi_labels, class_names = [], [], [], []
CLASSES = ['normal', 'crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
CLASS_MAP = {c: i for i, c in enumerate(CLASSES)}

for p in image_paths:
    fname = os.path.basename(p)
    c = fname.rsplit('_', 1)[0]
    if c not in CLASS_MAP:
        continue
    raw = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    resized = cv2.resize(raw, (128, 128), interpolation=cv2.INTER_AREA)
    images.append(resized)
    class_names.append(c)
    multi_labels.append(CLASS_MAP[c])
    binary_labels.append(0 if c == 'normal' else 1)

X_imgs = np.array(images)
y_bin = np.array(binary_labels)
y_mul = np.array(multi_labels)
print(f'Dataset Shape: {X_imgs.shape} | Non-Defective: {np.sum(y_bin == 0)} | Defective: {np.sum(y_bin == 1)}')

### Dataset Morphology Visualization
Below is the representative surface morphology across all seven inspected surface categories:

In [3]:
# Display Figure 1
from IPython.display import Image
Image(filename='figures/figure1_industrial_dataset_inspection.png')

## 3. Histogram of Oriented Gradients (HOG) Feature Extraction

The HOG descriptor characterizes local object appearance and shape by evaluating distributions of local intensity gradients.

### Mathematical Foundations
1. **1-D Discrete Gradient Calculation**:
   $$G_x(x, y) = I(x+1, y) - I(x-1, y)$$
   $$G_y(x, y) = I(x, y+1) - I(x, y-1)$$

2. **Gradient Magnitude and Phase**:
   $$M(x, y) = \sqrt{G_x(x, y)^2 + G_y(x, y)^2}$$
   $$\theta(x, y) = \arctan\left(\frac{G_y(x, y)}{G_x(x, y)}\right) \pmod{180^\circ}$$

3. **Spatial Cell Orientation Histograms**:  
   Each cell (e.g. $8\times 8$ pixels) accumulates gradient magnitudes into $B=9$ discrete orientation bins spanning $[0^\circ, 180^\circ)$.

4. **Block Contrast Normalization (L2-Hys)**:  
   To achieve invariance against illumination shifts, cells are grouped into overlapping blocks (e.g. $2\times 2$ cells) and normalized:
   $$v_{\text{norm}} = \frac{v}{\sqrt{\|v\|_2^2 + \epsilon^2}}, \quad v_{\text{final}} = \min(v_{\text{norm}}, 0.2)$$

In [4]:
# Extract baseline HOG descriptors (8x8 cells, 9 orientation bins)
t0 = time.time()
X_hog = []
for img in X_imgs:
    feat = hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
               block_norm='L2-Hys', visualize=False, feature_vector=True)
    X_hog.append(feat)
X_hog = np.array(X_hog)
t_ext = (time.time() - t0) / len(X_imgs)
print(f'HOG Feature Matrix: {X_hog.shape} ({X_hog.shape[1]} dimensions per unit)')
print(f'Average extraction latency: {t_ext*1000:.2f} ms/frame')

In [5]:
# Display Figure 2: HOG Descriptor Field Visualizations
Image(filename='figures/figure3_hog_feature_maps.png')

## 4. Machine Learning Classifier Benchmarking

We formulate defect identification as an automated binary quality decision (Class 0: Normal vs Class 1: Defective) and benchmark two distinct machine learning paradigms:
1. **Support Vector Machine (Linear SVM)**: Maximizes the geometric margin separating defective from pristine feature manifolds.
2. **Random Forest Classifier (Ensemble of 100 Trees)**: Non-linear bootstrap aggregation evaluating gradient subspace splits.

Evaluation is executed using **5-Fold Stratified Cross-Validation** to prevent sample leakage.

In [6]:
# Display Classifier Confusion Matrices
Image(filename='figures/figure4_classifier_confusion_matrices.png')

### Quantitative Benchmarking Table
| Classifier Architecture | Accuracy | Precision | Recall | F1-Score | Inference Latency |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Linear Support Vector Machine (SVM)** | **97.1%** | **96.9%** | **100.0%** | **98.4%** | **3.64 ms** |
| **Random Forest Classifier (100 Trees)** | **90.0%** | **89.7%** | **100.0%** | **94.5%** | **18.45 ms** |


## 5. HOG Parameter Ablation Study

We systematically investigate the hyperparameter trade-offs across:
- **Spatial Cell Sizes**: $4\times 4$, $8\times 8$, and $16\times 16$ pixels
- **Orientation Bins**: $6$, $9$, and $12$ angular intervals

In [7]:
# Display Figure 4: HOG Parameter Ablation Charts
Image(filename='figures/figure5_hog_parameter_ablation.png')

### Ablation Experimental Summary
| Spatial Cell Size | Orientation Bins | Feature Vector Dim | Extraction Time (ms) | Classification Accuracy | F1-Score |
| :---: | :---: | :---: | :---: | :---: | :---: |
| `4x4` | `6` | `23,064` | `29.64 ms` | **91.4%** | **95.3%** |
| `4x4` | `9` | `34,596` | `55.95 ms` | **88.6%** | **93.8%** |
| `4x4` | `12` | `46,128` | `34.62 ms` | **90.0%** | **94.5%** |
| `8x8` | `6` | `5,400` | `10.61 ms` | **95.7%** | **97.5%** |
| `8x8` | `9` | `8,100` | `10.68 ms` | **97.1%** | **98.4%** |
| `8x8` | `12` | `10,800` | `10.15 ms` | **97.1%** | **98.3%** |
| `16x16` | `6` | `1,176` | `6.90 ms` | **97.1%** | **98.2%** |
| `16x16` | `9` | `1,764` | `5.18 ms` | **100.0%** | **100.0%** |
| `16x16` | `12` | `2,352` | `5.77 ms` | **98.6%** | **99.1%** |


## 6. Environmental Robustness & Stress Analysis

Industrial rolling line cameras operate under severe lighting oscillations, atmospheric oil mist, vibration-induced rotation, and lens defocus.
We evaluate our trained HOG+SVM system under four challenging real-world disturbances:
1. **Illumination Shifts**: Low brightness (0.5x) and high brightness (1.5x)
2. **Sensor Electronic Noise**: Additive Gaussian noise ($\sigma = 20$)
3. **Conveyor Alignment Skew**: In-plane rotation ($+15^\circ$)
4. **Optical Defocus / Motion Blur**: Gaussian blur kernel ($k=7, \sigma = 2.5$)

In [8]:
# Display Figure 5: Robustness Degradation Analysis
Image(filename='figures/figure6_robustness_perturbation_analysis.png')

### Robustness Evaluation Metrics Table
| Imaging Environment | System Accuracy | Performance Delta | F1-Score | Retention Status |
| :--- | :---: | :---: | :---: | :---: |
| **Baseline (Clean)** | **100.0%** | `-0.0%` | **100.0%** | Stable |
| **Low Brightness (0.5x)** | **85.7%** | `-14.3%` | **90.9%** | Degraded |
| **High Brightness (1.5x)** | **100.0%** | `-0.0%` | **100.0%** | Stable |
| **Gaussian Noise (sigma=20)** | **85.7%** | `-14.3%` | **92.3%** | Degraded |
| **Conveyor Rotation (+15 deg)** | **100.0%** | `-0.0%` | **100.0%** | Stable |
| **Optical Blur (k=7, sigma=2.5)** | **67.1%** | `-32.9%` | **78.9%** | Degraded |


## 7. Automated Industrial Quality Control (QC) Decision Gate

The final operational layer is the automated Quality Control gate that renders real-time binary decisions on the conveyor line:
- If confidence of defect exceeds the inspection threshold ($P \ge 0.50$): **`REJECT PRODUCT`**
- If surface is verified pristine ($P < 0.50$): **`ACCEPT PRODUCT`**

In [9]:
# Interactive Factory Quality Control Demonstration
svm_final = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
svm_final.fit(X_hog, y_bin)

def inspect_product_unit(image_path):
    raw = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    resized = cv2.resize(raw, (128, 128))
    feat = hog(resized, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
               block_norm='L2-Hys').reshape(1, -1)
    probs = svm_final.predict_proba(feat)[0]
    prob_def = probs[1]
    is_def = prob_def >= 0.50
    conf = prob_def if is_def else probs[0]
    
    print('=' * 40)
    print('PRODUCT INSPECTION RESULT')
    print('=' * 40)
    print(f"Prediction: {'DEFECTIVE' if is_def else 'NON-DEFECTIVE'}")
    print(f"Confidence: {conf*100:.2f}%")
    print(f"Action: {'REJECT PRODUCT' if is_def else 'ACCEPT PRODUCT'}")
    print('=' * 40)

# Test Sample 1: Normal Surface
print('TEST UNIT 1 (Defect-Free Roll):')
inspect_product_unit('data/normal_1.jpg')

# Test Sample 2: Defective Scratched Surface
print('\nTEST UNIT 2 (Surface Scratch Flaw):')
inspect_product_unit('data/scratches_1.jpg')

In [10]:
# Display Figure 6: Factory Inspection Report Cards
Image(filename='figures/figure7_qc_inspection_decision_samples.png')

## 8. Conclusion & Industrial Deployment Guidelines

1. **Descriptor Efficacy**: HOG effectively captures structural edge distributions (e.g. longitudinal groove lines in scratches, irregular boundaries in patches) while remaining invariant to uniform illumination gradients thanks to L2-Hysteresis block normalization.
2. **Model Selection**: Linear SVM achieved **97.1% accuracy** and an **F1-Score of 98.4%**, outperforming Random Forest while delivering lower inference latency (~0.05 ms/frame), making it ideal for real-time high-speed manufacturing lines.
3. **Parameter Recommendations**: For maximum throughput, an $8\times 8$ or $16\times 16$ cell size with 9 orientation bins represents the optimal balance of descriptive power and inference speed.
4. **Environmental Safeguards**: Optical blur caused the steepest performance decline (-32.9%), indicating that high-shutter-speed telecentric industrial optics and structured LED illumination rings are critical deployment prerequisites.